# APPLICATION OF XGBOOST FROM APRIL 2026

In [16]:
# ============================================================
# 🔍 PREDICT SECTION & CLUSTER — Device: LL1801
# ============================================================

import pandas as pd
import numpy as np
import re, pickle, os, warnings
warnings.filterwarnings('ignore')

# ── 1. LOAD MODELS & ENCODERS ────────────────────────────────
SAVE_DIR = r'C:\Users\sitisyaziyah\Documents\Device_Identifier\Section\XGBoost16042026'

with open(os.path.join(SAVE_DIR, 'model_section.pkl'), 'rb') as f:
    model_section = pickle.load(f)

with open(os.path.join(SAVE_DIR, 'model_cluster.pkl'), 'rb') as f:
    model_cluster = pickle.load(f)

with open(os.path.join(SAVE_DIR, 'label_encoders.pkl'), 'rb') as f:
    enc           = pickle.load(f)

le_customer    = enc['customer']
le_project     = enc['project']
le_prefix      = enc['prefix']
le_suffix_lt   = enc['suffix_lt']
le_suffix_last = enc['suffix_last']
le_section     = enc['section']
le_cluster     = enc['cluster']
feature_cols   = enc['feature_columns']

print(f"✅ Models loaded | Sections: {list(le_section.classes_)} | Clusters: {list(le_cluster.classes_)}")

# ── 2. FEATURE ENGINEERING HELPERS ──────────────────────────
def extract_numeric_block(did):
    m = re.search(r'\d+', str(did))
    return int(m.group()) if m else -1

def extract_prefix(did):
    m = re.match(r'^([A-Za-z]+)', str(did))
    return m.group(1).upper() if m else ''

def extract_suffix_lt(did):
    m = re.search(r'\d+([A-Za-z]*)$', str(did))
    return m.group(1).upper() if m else ''

def extract_suffix_full(did):
    m = re.search(r'\d+(.*)$', str(did))
    return m.group(1) if m else ''

def safe_encode(encoder, value, default=0):
    try:
        return int(encoder.transform([value])[0])
    except ValueError:
        print(f"  ⚠ Unseen label '{value}' → using default={default}")
        return default

import pandas as pd

df_oiltek_num = pd.read_pickle(
    r"C:\Users\sitisyaziyah\Documents\Device_Identifier\Section\XGBoost16042026\df_oiltek_num.pkl"
)

# ── 3. INPUT ─────────────────────────────────────────────────
device_id = 'LL510'
customer  = 'OILTEK'
project   = 'A1706'

# ── 4. COMPUTE numeric_block_rank (PURE NUMERIC COMPARISON) ──
# ── 4. COMPUTE numeric_block_rank (FROM df_oiltek_num) ───────

numeric_block = extract_numeric_block(device_id)

# Filter relevant subset
df_ref = df_oiltek_num[
    (df_oiltek_num["CUSTOMER"] == customer) &
    (df_oiltek_num["PROJECT"] == "A1706")
]

if not df_ref.empty:
    # Ensure numeric_block column is clean
    df_ref = df_ref.copy()
    df_ref["numeric_block"] = pd.to_numeric(df_ref["numeric_block"], errors="coerce")

    # Get sorted unique numeric blocks
    unique_blocks = sorted(df_ref["numeric_block"].dropna().unique())

    # PURE ranking
    numeric_block_rank = sum(b <= numeric_block for b in unique_blocks)

    # Edge case: if all values are bigger
    if numeric_block_rank == 0:
        numeric_block_rank = 1

    print(f"  ✅ Numeric-based rank (df reference): {numeric_block} → rank {numeric_block_rank}")

else:
    numeric_block_rank = 1
    print("  ⚠ No reference data found — using rank=1 (fallback)")

✅ Models loaded | Sections: ['SECTION 1', 'SECTION 2', 'SECTION 3', 'SECTION 4', 'SECTION 5', 'SECTION 6'] | Clusters: ['CLUSTER 1', 'CLUSTER 2', 'CLUSTER 3', 'CLUSTER 4', 'CLUSTER 5']
  ✅ Numeric-based rank (df reference): 510 → rank 6


In [17]:
# ── 5. BUILD FEATURE VECTOR ──────────────────────────────────
pfx        = extract_prefix(device_id)
sfx_lt     = extract_suffix_lt(device_id)
sfx_full   = extract_suffix_full(device_id)
sfx_last   = sfx_full[-1] if sfx_full else ''


# ── 5. BUILD FEATURE VECTOR (NUMERIC-ONLY VERSION) ───────────
fv = {
    'customer_enc'       : safe_encode(le_customer, customer),
    'project_enc'        : safe_encode(le_project, project),
    'numeric_block'      : numeric_block,
    'numeric_block_rank' : numeric_block_rank,
    'device_id_length'   : len(device_id),
    'has_numeric'        : int(numeric_block != -1),
    'equip_id_length'    : len(device_id),
    'equip_id_digit_count': len(re.findall(r'\d', device_id)),
    'equip_id_letter_count': len(re.findall(r'[A-Za-z]', device_id)),
}

X_new = pd.DataFrame([fv])

X_new = pd.DataFrame([fv])

# Ensure all expected columns exist
for col in feature_cols:
    if col not in X_new.columns:
        X_new[col] = 0

# Reorder columns
X_new = X_new[feature_cols]


In [15]:
# ── 6. PREDICT ──────────────────────────────────────────────
assert list(X_new.columns) == list(feature_cols), "❌ Feature mismatch with training!"

try:
    sec_proba = model_section.predict_proba(X_new)[0]
    clu_proba = model_cluster.predict_proba(X_new)[0]
except Exception as e:
    print(f"❌ Prediction failed: {e}")
    raise

sec_pred = le_section.classes_[np.argmax(sec_proba)]
clu_pred = le_cluster.classes_[np.argmax(clu_proba)]

sec_conf = np.max(sec_proba)
clu_conf = np.max(clu_proba)


# ── 7. DISPLAY RESULTS ──────────────────────────────────────
print("\n" + "=" * 65)
print(f"  PREDICTION RESULT — {device_id} | {customer} | {project}")
print("=" * 65)
print(f"  ✅ Section  →  {sec_pred}   ({sec_conf:.2%})")
print(f"  ✅ Cluster  →  {clu_pred}   ({clu_conf:.2%})")

print("\n  SECTION probabilities:")
sec_df = pd.DataFrame({'Section': le_section.classes_, 'Prob': sec_proba})
sec_df = sec_df.sort_values('Prob', ascending=False).reset_index(drop=True)

for _, row in sec_df.iterrows():
    bar = '█' * int(row['Prob'] * 40)
    tag = '<< PREDICTED' if row['Section'] == sec_pred else ''
    print(f"    {row['Section']:<12} {row['Prob']:6.2%}  {bar} {tag}")

print("\n  CLUSTER probabilities:")
clu_df = pd.DataFrame({'Cluster': le_cluster.classes_, 'Prob': clu_proba})
clu_df = clu_df.sort_values('Prob', ascending=False).reset_index(drop=True)

for _, row in clu_df.iterrows():
    bar = '█' * int(row['Prob'] * 40)
    tag = '<< PREDICTED' if row['Cluster'] == clu_pred else ''
    print(f"    {row['Cluster']:<12} {row['Prob']:6.2%}  {bar} {tag}")

print("\n" + "=" * 65)


  PREDICTION RESULT — LL510 | OILTEK | A1706
  ✅ Section  →  SECTION 1   (65.80%)
  ✅ Cluster  →  CLUSTER 1   (99.48%)

  SECTION probabilities:
    SECTION 1    65.80%  ██████████████████████████ << PREDICTED
    SECTION 2    27.50%  ███████████ 
    SECTION 3     3.32%  █ 
    SECTION 4     1.17%   
    SECTION 5     1.16%   
    SECTION 6     1.05%   

  CLUSTER probabilities:
    CLUSTER 1    99.48%  ███████████████████████████████████████ << PREDICTED
    CLUSTER 3     0.21%   
    CLUSTER 4     0.16%   
    CLUSTER 5     0.08%   
    CLUSTER 2     0.07%   



In [22]:
print(df_oiltek_num[(df_oiltek_num["PROJECT"] == "A1706")& (df_oiltek_num["SECTION"] == "SECTION 1") & (df_oiltek_num["CLUSTER"] == "CLUSTER 1")])

  CUSTOMER PROJECT    SECTION    CLUSTER DEVICE_ID  numeric_block
0   OILTEK   A1706  SECTION 1  CLUSTER 1      A500            500
1   OILTEK   A1706  SECTION 1  CLUSTER 1     HT500            500
2   OILTEK   A1706  SECTION 1  CLUSTER 1    PT500A            500
3   OILTEK   A1706  SECTION 1  CLUSTER 1    PV500B            500
4   OILTEK   A1706  SECTION 1  CLUSTER 1     TE500            500
